# Investment Track ML

A reproducible notebook for ETF and stock direction research, developed as a supporting tool for the CaseKey Investment Track. It compares three classifiers against training-frequency baselines, audits time-series splits, and exports evidence for human review.

**Research scores are uncalibrated. They do not estimate return magnitude, demonstrate trading profit, or authorize orders.**

This public edition contains the research workflow. The original account screenshots, personal portfolio balances, policy documents, correspondence, and meeting materials are excluded.

## Run the notebook

Run all cells in order. The default is the historical November 10, 2026 review, using completed sessions through November 9. Change `REVIEW_DATE` in the configuration cell for another review. It is the **exclusive** download end. Runtime downloads may differ from the original data after provider revisions.

The installation cell is for notebook environments such as Colab. For local use, install `requirements.txt` first. Each execution writes a separate output directory.


In [ ]:
%pip -q install numpy==2.3.5 pandas==3.1.1 scikit-learn==1.9.1 xgboost==3.4.1 yfinance==1.8.0 pandas_market_calendars==5.6.0 matplotlib==3.11.2 seaborn==0.13.2 ipython==9.18.1


In [ ]:
import json
import math
import time
import urllib.parse
import urllib.request
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

from sklearn.base import clone
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    confusion_matrix,
    roc_auc_score,
)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from xgboost import XGBClassifier

pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 180)
sns.set_theme(style="whitegrid")
RANDOM_STATE = 42
print("Environment ready")
import yfinance as yf
import pandas_market_calendars as mcal
import hashlib
import importlib.metadata
import shutil
from sklearn.metrics import brier_score_loss, log_loss
from IPython.display import display
from sklearn.model_selection import TimeSeriesSplit

import os
from datetime import date


## Research configuration

The example universe contains 12 ETFs and seven stocks. Inclusion is for research and does not establish eligibility under any investment policy. The optional November 13 cutoff adds a four-session target for the default review. Five- and 20-session targets extend beyond that cutoff. The cutoff is historical project context, not a current trading deadline.

Targets start at the last completed close and end at the corresponding future session close. They do not measure returns available from entering at an intraday quote.


In [ ]:
START = '2016-01-01'
REVIEW_DATE = os.environ.get('ITML_REVIEW_DATE', '2026-11-10')  # exclusive end
if date.fromisoformat(REVIEW_DATE) > pd.Timestamp.now(tz='America/New_York').date():
    raise ValueError('Review date must not include future or incomplete sessions')
CUTOFF = '2026-11-13'
CUTOFF_SOURCE = 'Historical CaseKey project configuration; not a live deadline'
calendar = mcal.get_calendar('NYSE')
calendar_end = max(pd.Timestamp(REVIEW_DATE), pd.Timestamp(CUTOFF)) + pd.Timedelta(days=90)
sessions = calendar.valid_days(start_date=START, end_date=calendar_end).tz_localize(None)
EXPECTED_LAST = sessions[sessions < pd.Timestamp(REVIEW_DATE)].max()
CUTOFF_HORIZON = int(((sessions > EXPECTED_LAST) & (sessions <= pd.Timestamp(CUTOFF))).sum())
HORIZONS = tuple(sorted(set([5, 20] + ([CUTOFF_HORIZON] if CUTOFF_HORIZON > 0 else []))))
UNIVERSE = dict.fromkeys(['SPY','IVV','XLK','XLC','XLY','XLV','XLI','ACWX','VEU','AGG','BND','SCHZ'], 'etf')
UNIVERSE.update(dict.fromkeys(['AAPL','MSFT','GOOGL','META','AMZN','TSLA','NFLX'], 'stocks'))
RUN_ID = pd.Timestamp.now(tz='UTC').strftime('%Y%m%dT%H%M%S%fZ')
OUTPUT_ROOT = Path(os.environ.get('ITML_OUTPUT_ROOT', 'outputs')).resolve()
RUN_DIR = OUTPUT_ROOT / ('review_' + REVIEW_DATE) / RUN_ID
DATA_DIR = RUN_DIR / 'prices'
DATA_DIR.mkdir(parents=True, exist_ok=True)
yf.set_tz_cache_location(str(RUN_DIR / 'yfinance_cache'))
FEATURES = ['ret_1','ret_5','ret_20','vol_20','vol_60','ma_20_gap','ma_50_gap',
            'volume_z_20','range_pct','rsi_14','spy_ret_5','spy_vol_20']
future_sessions = sessions[sessions > EXPECTED_LAST]
horizon_table = pd.DataFrame([
    dict(horizon_sessions=h, starts_at_close=str(EXPECTED_LAST.date()),
         ends_at_close=str(future_sessions[h-1].date()),
         within_cutoff=bool(future_sessions[h-1] <= pd.Timestamp(CUTOFF)))
    for h in HORIZONS])
print('Expected completed session:', EXPECTED_LAST.date(), '| cutoff:', CUTOFF)
print('Run directory:', RUN_DIR)
display(horizon_table)


## Download and validate adjusted prices

Yahoo Finance data is downloaded using `yfinance` with `auto_adjust=True`. These are research prices rather than broker execution prices or an official performance record. Every symbol must pass before models run: required values, positive finite OHLC, nonnegative volume, unique dates, a complete exchange-session sequence through the expected close, OHLC consistency, sufficient history, and large-jump review.

The OHLC tolerance only accommodates floating-point rounding. Adjusted moves above 45% stop the run for manual corporate-action review; they are never replaced with zero. XLC history before its June 19, 2018 listing is rejected. Downloads, timestamps and SHA-256 hashes are saved locally.


In [ ]:
raw, audit_rows, download_errors = {}, [], []
for symbol in UNIVERSE:
    try:
        frame = yf.download(symbol, start=START, end=REVIEW_DATE, auto_adjust=True,
                            actions=False, progress=False, threads=False,
                            multi_level_index=False, timeout=30)
        if frame is None or frame.empty:
            raise ValueError('Empty price response')
        frame = frame.rename_axis('date').reset_index()
        frame.columns = [str(c).lower() for c in frame.columns]
        frame['date'] = pd.to_datetime(frame['date']).dt.tz_localize(None).dt.normalize()
        frame = frame.sort_values('date').reset_index(drop=True)
        required = ['date','open','high','low','close','volume']
        if (frame[required].isna().any().any() or frame.date.duplicated().any()
            or not np.isfinite(frame[['open','high','low','close','volume']].to_numpy()).all()):
            raise ValueError('Missing/non-finite values or duplicate dates')
        if (frame[['open','high','low','close']] <= 0).any().any() or (frame.volume < 0).any():
            raise ValueError('Invalid prices or volume')
        expected_dates = sessions[(sessions >= frame.date.min()) & (sessions <= EXPECTED_LAST)]
        if list(frame.date) != list(expected_dates):
            raise ValueError('Missing, extra, or stale exchange sessions; last=' + str(frame.date.max()))
        tolerance = frame.close.abs() * 1e-10
        bad_range = (frame.high + tolerance < frame.low) | (frame.high + tolerance < frame[['open','close']].max(axis=1)) | (frame.low - tolerance > frame[['open','close']].min(axis=1))
        if bad_range.any():
            frame.loc[bad_range].to_csv(RUN_DIR / (symbol + '_rejected_ohlc.csv'), index=False)
            raise ValueError('Inconsistent OHLC range beyond floating-point tolerance')
        if symbol == 'XLC' and frame.date.min() < pd.Timestamp('2018-06-19'):
            raise ValueError('XLC history predates its verified exchange listing')
        if len(frame) < 1500:
            raise ValueError('Insufficient history')
        if frame.close.pct_change().abs().gt(0.45).any():
            raise ValueError('Adjusted price jump >45%; manual corporate-action review required')
        frame['symbol'] = symbol
        path = DATA_DIR / (symbol + '_adjusted.csv')
        frame.to_csv(path, index=False)
        audit_rows.append(dict(symbol=symbol, rows=len(frame), first=str(frame.date.min().date()),
                               last=str(frame.date.max().date()), source='Yahoo Finance via yfinance; auto_adjust=True',
                               fetched_utc=pd.Timestamp.now(tz='UTC').isoformat(),
                               sha256=hashlib.sha256(path.read_bytes()).hexdigest()))
        raw[symbol] = frame
        print(symbol, len(frame), str(frame.date.max().date()), flush=True)
    except Exception as exc:
        download_errors.append(dict(symbol=symbol, error=str(exc)))
        print('DATA ERROR:', symbol, str(exc), flush=True)
data_audit = pd.DataFrame(audit_rows)
data_audit.to_csv(RUN_DIR / 'Data_Audit.csv', index=False)
if download_errors:
    pd.DataFrame(download_errors).to_csv(RUN_DIR / 'Data_Errors.csv', index=False)
    raise RuntimeError('Price verification failed. No current signals produced. See Data_Errors.csv.')
assert len(raw) == len(UNIVERSE)


## Causal features and forward labels

Features use only the observation date and earlier prices: 1/5/20-session returns, 20/60-session annualized volatility, 20/50-session moving-average gaps, rolling log-volume z-score, daily range, 14-session simple-average RSI, and SPY return/volatility context. Data becomes available after that session closes.

Labels use future adjusted close-to-close returns and remain missing until the full horizon has matured. A model predicts whether the return is positive, not its size. Adjusted history can be revised; the source is not a point-in-time data archive.


In [ ]:
def rsi(series, window=14):
    delta = series.diff()
    gain = delta.clip(lower=0).rolling(window).mean()
    loss = -delta.clip(upper=0).rolling(window).mean()
    relative_strength = gain / loss.replace(0, 1e-12)
    return 100 - 100 / (1 + relative_strength)


def build_research_index(close):
    adjusted_return = close.pct_change()
    return close / close.iloc[0] * 100, adjusted_return, pd.Series(False, index=close.index)


def make_features(df, spy_context):
    x = df.copy()
    x["research_index"], clean_return, x["possible_corporate_action"] = build_research_index(x["close"])
    x["ret_1"] = clean_return
    x["ret_5"] = x["research_index"].pct_change(5)
    x["ret_20"] = x["research_index"].pct_change(20)
    x["vol_20"] = clean_return.rolling(20).std() * math.sqrt(252)
    x["vol_60"] = clean_return.rolling(60).std() * math.sqrt(252)
    x["ma_20_gap"] = x["research_index"] / x["research_index"].rolling(20).mean() - 1
    x["ma_50_gap"] = x["research_index"] / x["research_index"].rolling(50).mean() - 1

    log_volume = np.log1p(x["volume"])
    x["volume_z_20"] = (
        (log_volume - log_volume.rolling(20).mean()) /
        log_volume.rolling(20).std()
    )
    x["range_pct"] = (x["high"] - x["low"]) / x["close"].replace(0, np.nan)
    x["rsi_14"] = rsi(x["research_index"])
    for horizon in HORIZONS:
        x[f"label_end_{horizon}"] = x["date"].shift(-horizon)
        x[f"forward_{horizon}"] = x["research_index"].shift(-horizon) / x["research_index"] - 1

    market = spy_context[["date", "ret_5", "vol_20"]].rename(columns={
        "ret_5": "spy_ret_5",
        "vol_20": "spy_vol_20",
    })
    return x.merge(market, on="date", how="left")



In [ ]:
spy = raw["SPY"].copy()
spy["research_index"], spy_clean_return, _ = build_research_index(spy["close"])
spy["ret_5"] = spy["research_index"].pct_change(5)
spy["vol_20"] = spy_clean_return.rolling(20).std() * math.sqrt(252)

featured = {ticker: make_features(frame, spy) for ticker, frame in raw.items()}
featured["AAPL"][["date", "close"] + FEATURES].tail(3)


## Model validation
Reserve the latest 15% of labeled observations as a historical holdout. On the earlier
history, use three expanding validation folds, with a horizon-sized gap. Select the
model with the highest mean validation Brier improvement over each fold's constant
training-frequency baseline. Fit it on pre-holdout data, evaluate the holdout, then
refit on all matured labels to estimate the latest score.

Every split asserts that training labels end strictly before validation/test starts.
The saved split audit exposes those dates. No shuffling. This is repeated historical
research, not an untouched prospective test. Labels overlap, funds overlap, and many
symbols/horizons are examined: observations and results are not independent.


In [ ]:
def model_library():
    return {
        "Logistic Regression": Pipeline([
            ("scale", StandardScaler()),
            ("model", LogisticRegression(
                max_iter=3000,
                class_weight=None,
                random_state=RANDOM_STATE,
            )),
        ]),
        "Gradient Boosting": GradientBoostingClassifier(
            n_estimators=80,
            learning_rate=0.04,
            max_depth=2,
            min_samples_leaf=25,
            subsample=0.8,
            random_state=RANDOM_STATE,
        ),
        "XGBoost": XGBClassifier(
            n_estimators=160,
            max_depth=3,
            learning_rate=0.03,
            min_child_weight=8,
            subsample=0.8,
            colsample_bytree=0.8,
            reg_lambda=2.0,
            eval_metric="logloss",
            tree_method="hist",
            random_state=RANDOM_STATE,
            n_jobs=2,
        ),
    }


def safe_auc(y_true, probability):
    if pd.Series(y_true).nunique() < 2:
        return np.nan
    return roc_auc_score(y_true, probability)


def classification_metrics(y_true, probability):
    prediction = (probability >= 0.50).astype(int)
    positive_rate = float(np.mean(y_true))
    return {
        "roc_auc": safe_auc(y_true, probability),
        "brier": brier_score_loss(y_true, probability),
        "log_loss": log_loss(y_true, probability, labels=[0, 1]),
        "accuracy": accuracy_score(y_true, prediction),
        "balanced_accuracy": balanced_accuracy_score(y_true, prediction),
        "positive_rate": positive_rate,
        "holdout_majority_accuracy_descriptive": max(positive_rate, 1 - positive_rate),
    }


In [ ]:
split_rows = []

def metrics_with_baseline(y, probabilities, training_y):
    metrics = classification_metrics(y, probabilities)
    base_rate = float(np.mean(training_y))
    baseline = np.full(len(y), base_rate)
    metrics.update(baseline_brier=brier_score_loss(y, baseline),
                   baseline_log_loss=log_loss(y, baseline, labels=[0, 1]),
                   baseline_accuracy=accuracy_score(y, (baseline >= 0.5).astype(int)),
                   reference_base_rate=base_rate)
    metrics['brier_improvement'] = metrics['baseline_brier'] - metrics['brier']
    return metrics, baseline

def check_split(symbol, horizon, stage, train, evaluation):
    end = train[f'label_end_{horizon}'].max()
    start = evaluation.date.min()
    assert end < start, f'Label leakage: {symbol}/{horizon}/{stage}'
    if train.target.nunique() < 2:
        raise ValueError('Training fold has only one target class')
    split_rows.append(dict(symbol=symbol, horizon=horizon, stage=stage,
        train_start=str(train.date.min().date()), train_end=str(train.date.max().date()),
        last_train_label_end=str(end.date()), evaluation_start=str(start.date()),
        evaluation_end=str(evaluation.date.max().date()),
        train_rows=len(train), evaluation_rows=len(evaluation), purged=True))

def analyze_symbol_horizon(symbol, df, horizon):
    target_column = f'forward_{horizon}'
    data = df.replace([np.inf, -np.inf], np.nan).dropna(subset=FEATURES + [target_column]).copy()
    data['target'] = (data[target_column] > 0).astype(int)
    test_start = int(len(data) * 0.85)
    development = data.iloc[:test_start-horizon]
    test = data.iloc[test_start:]
    if len(development) < 750 or len(test) < 100:
        raise ValueError('Insufficient validation/holdout history')
    check_split(symbol, horizon, 'holdout', development, test)
    validation_rows = []
    for fold, (train_idx, valid_idx) in enumerate(TimeSeriesSplit(n_splits=3, gap=horizon).split(development), 1):
        train, valid = development.iloc[train_idx], development.iloc[valid_idx]
        check_split(symbol, horizon, f'validation_{fold}', train, valid)
        for model_name, model in model_library().items():
            model.fit(train[FEATURES], train.target)
            probabilities = model.predict_proba(valid[FEATURES])[:, 1]
            metrics, _ = metrics_with_baseline(valid.target, probabilities, train.target)
            validation_rows.append(dict(symbol=symbol, horizon_days=horizon,
                model=model_name, period=f'validation_{fold}',
                start=str(valid.date.min().date()), end=str(valid.date.max().date()),
                rows=len(valid), **metrics))
    selection = pd.DataFrame(validation_rows).groupby('model').brier_improvement.mean()
    selected_name = str(selection.idxmax())
    selected_model = clone(model_library()[selected_name])
    selected_model.fit(development[FEATURES], development.target)
    test_probability = selected_model.predict_proba(test[FEATURES])[:, 1]
    test_metrics, baseline_probability = metrics_with_baseline(test.target, test_probability, development.target)
    test_metrics['validation_brier_improvement'] = float(selection.loc[selected_name])
    test_row = dict(symbol=symbol, horizon_days=horizon, model=selected_name,
        period='historical_holdout', start=str(test.date.min().date()),
        end=str(test.date.max().date()), rows=len(test), **test_metrics)
    final_model = clone(model_library()[selected_name])
    final_model.fit(data[FEATURES], data.target)
    latest = df.dropna(subset=FEATURES).iloc[-1:]
    assert latest.date.iloc[0] == EXPECTED_LAST
    current_probability = float(final_model.predict_proba(latest[FEATURES])[:, 1][0])
    return dict(selected_model=selected_name, current_probability=current_probability,
        latest_date=str(latest.date.iloc[0].date()), latest_close=float(latest.close.iloc[0]),
        baseline_probability=baseline_probability, test_metrics=test_metrics,
        validation_rows=validation_rows, test_row=test_row,
        test_truth=test.target.to_numpy(), test_probability=test_probability)


## Interpret research flags
The gate requires positive mean validation Brier improvement, holdout AUC >=0.55,
and improvement in both Brier and log loss over the pre-holdout constant baseline.
It is a declared research heuristic, not evidence of statistical significance or
net trading profit. The holdout is used to create these flags, so flags require future
validation. Score deviations of +/-8 percentage points are descriptive only.


In [ ]:
def research_signal(score, metrics):
    if (metrics['validation_brier_improvement'] <= 0 or pd.isna(metrics['roc_auc']) or metrics['roc_auc'] < 0.55
        or metrics['brier'] >= metrics['baseline_brier']
        or metrics['log_loss'] >= metrics['baseline_log_loss']):
        return 'Insufficient model evidence'
    delta = score - metrics['reference_base_rate']
    if delta >= 0.08:
        return 'Research candidate only'
    if delta <= -0.08:
        return 'Review downside; not a sell instruction'
    return 'Neutral'


In [ ]:
signal_rows, metric_rows, diagnostics = [], [], {}
for symbol, frame in featured.items():
    signal_row = dict(symbol=symbol, asset_type='ETF' if UNIVERSE[symbol] == 'etf' else 'Stock')
    for horizon in HORIZONS:
        result = analyze_symbol_horizon(symbol, frame, horizon)
        metrics = result['test_metrics']
        signal_row.update(as_of=result['latest_date'], adjusted_close=result['latest_close'])
        signal_row[f'prob_up_{horizon}d'] = result['current_probability']
        signal_row[f'model_{horizon}d'] = result['selected_model']
        for key in ['roc_auc','brier','baseline_brier','log_loss','baseline_log_loss','reference_base_rate']:
            signal_row[f'test_{key}_{horizon}d'] = metrics[key]
        signal_row[f'signal_{horizon}d'] = research_signal(result['current_probability'], metrics)
        metric_rows.extend(result['validation_rows'])
        metric_rows.append(result['test_row'])
        diagnostics[(symbol, horizon)] = result
    signal_rows.append(signal_row)
    print('Finished', symbol, flush=True)
signals = pd.DataFrame(signal_rows).sort_values('symbol').reset_index(drop=True)
validation = pd.DataFrame(metric_rows)
display_columns = ['symbol','as_of'] + [c for h in HORIZONS for c in
                   [f'prob_up_{h}d',f'test_roc_auc_{h}d',f'signal_{h}d']]
display(signals[display_columns])


## Diagnostic charts

The first chart displays the uncalibrated latest model score; the second shows historical holdout AUC. A high score does not correct weak holdout discrimination. The confusion matrix uses a fixed 0.50 classification threshold and is descriptive.


In [ ]:
PLOT_HORIZON = CUTOFF_HORIZON if CUTOFF_HORIZON > 0 else 5
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

ordered = signals.sort_values(f"prob_up_{PLOT_HORIZON}d")
axes[0].barh(ordered["symbol"], ordered[f"prob_up_{PLOT_HORIZON}d"], color="#3572A5")
axes[0].axvline(0.50, color="black", linestyle="--", linewidth=1)
axes[0].set_title(f"Uncalibrated score: positive {PLOT_HORIZON}-session return")
axes[0].set_xlabel("Model score")

ordered_auc = signals.sort_values(f"test_roc_auc_{PLOT_HORIZON}d")
axes[1].barh(ordered_auc["symbol"], ordered_auc[f"test_roc_auc_{PLOT_HORIZON}d"], color="#2A9D8F")
axes[1].axvline(0.50, color="black", linestyle="--", linewidth=1, label="Random")
axes[1].axvline(0.55, color="#E76F51", linestyle=":", linewidth=2, label="Research gate")
axes[1].set_title(f"Historical holdout AUC: {PLOT_HORIZON} sessions")
axes[1].set_xlabel("ROC AUC")
axes[1].legend()

plt.tight_layout()
plt.savefig(RUN_DIR / 'Model_Diagnostics.png', dpi=160, bbox_inches='tight')
plt.show()


In [ ]:
# Change these two values to inspect any ticker/horizon.
DIAGNOSTIC_SYMBOL = "MSFT"
DIAGNOSTIC_HORIZON = PLOT_HORIZON

d = diagnostics[(DIAGNOSTIC_SYMBOL, DIAGNOSTIC_HORIZON)]
cm = confusion_matrix(d["test_truth"], d["test_probability"] >= 0.50)

print("Selected model:", d["selected_model"])
print("Uncalibrated score:", f"{d['current_probability']:.1%}")
print("Historical holdout metrics:", d["test_metrics"])

sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", cbar=False)
plt.title(f"{DIAGNOSTIC_SYMBOL} {DIAGNOSTIC_HORIZON}D — test confusion matrix")
plt.xlabel("Predicted class")
plt.ylabel("Actual class")
plt.show()


## Human research checklist

Complete this checklist with dated primary-source evidence and your own investment policy before considering a decision. Blank fields are deliberately unreviewed. The price model does not verify ETF exposure, assets under management, costs, tracking difference, liquidity, company cash flows, leverage, valuation, or investment thesis.

This public template does not implement policy compliance, portfolio sizing, drawdown monitoring, performance attribution, or trade execution.


In [ ]:
review = pd.DataFrame({'symbol': list(UNIVERSE),
    'asset_type': ['ETF' if UNIVERSE[s] == 'etf' else 'Stock' for s in UNIVERSE]})
for column in ['policy_eligibility', 'required_exposure', 'company_fundamentals',
               'fund_aum_costs_tracking_liquidity', 'valuation_evidence', 'thesis',
               'thesis_breaker', 'portfolio_risk_review', 'source_links_and_dates']:
    review[column] = ''
review['human_decision'] = 'UNREVIEWED - no order'
display(review)


In [ ]:
trade_memos = signals[['symbol'] + [c for h in HORIZONS for c in
                     [f'prob_up_{h}d',f'test_roc_auc_{h}d',f'signal_{h}d']]].copy()
trade_memos['planned_action'] = 'WAIT - manual investment-policy review required'
for column in ['planned_weight_pct','my_thesis','key_facts_checked','valuation_evidence',
               'main_risk','exit_trigger','source_links','final_decision_and_date']:
    trade_memos[column] = ''
display(trade_memos)

## Export the research evidence

Each run saves signal scores, all model validation rows, split boundaries, the manual-review template, memo template, method notes, data audit, horizon dates, dependency versions, downloaded prices, and a ZIP bundle. Generated outputs and downloaded prices are ignored by Git. The ZIP is a local research bundle, not an automatic public release. Check data-provider terms before redistribution.


In [ ]:
method_notes = pd.DataFrame([
    ('Data','Fresh Yahoo Finance adjusted OHLC; strict full-session validation; not broker marks'),
    ('Window',f'{START} through {EXPECTED_LAST.date()}; exclusive end {REVIEW_DATE}'),
    ('Cutoff',f'{CUTOFF}; source: {CUTOFF_SOURCE}'),
    ('Targets',str(HORIZONS) + '; positive close-to-close adjusted return, not return magnitude'),
    ('Validation','3 expanding folds; horizon gap; selection by mean Brier improvement'),
    ('Holdout','Latest 15%, horizon purge, then refit on matured labels; reused research, not prospective proof'),
    ('Gate','Positive validation improvement; holdout AUC >=0.55 and Brier/log-loss improve'),
    ('Limitations','Overlapping labels and holdings; multiple comparisons; uncalibrated scores; no costed strategy backtest'),
    ('Sources','https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.TimeSeriesSplit.html | https://scikit-learn.org/stable/modules/calibration.html | https://ranaroussi.github.io/yfinance/reference/api/yfinance.download.html'),
], columns=['item','specification'])
tables = dict(ML_Signals=signals, Validation=validation, Split_Audit=pd.DataFrame(split_rows),
    Human_Review=review, Trade_Memos=trade_memos, Method=method_notes,
    Data_Audit=data_audit, Horizons=horizon_table)
for name, table in tables.items():
    table.to_csv(RUN_DIR / (name + '.csv'), index=False)
versions = {p: importlib.metadata.version(p) for p in
    ['numpy','pandas','scikit-learn','xgboost','yfinance','pandas_market_calendars']}
(RUN_DIR / 'Run_Manifest.json').write_text(json.dumps(dict(
    review_date=REVIEW_DATE, last_session=str(EXPECTED_LAST.date()), cutoff=CUTOFF,
    cutoff_source=CUTOFF_SOURCE, horizons=HORIZONS, universe=list(UNIVERSE), versions=versions,
    note='Research only. No account data included. Human policy review is uncompleted.'), indent=2))
import zipfile
archive = str(RUN_DIR) + '.zip'
with zipfile.ZipFile(archive, 'w', zipfile.ZIP_DEFLATED) as bundle:
    for path in sorted(RUN_DIR.rglob('*')):
        if path.is_file() and 'yfinance_cache' not in path.relative_to(RUN_DIR).parts:
            bundle.write(path, path.relative_to(RUN_DIR))
print('Saved CSV research bundle:', archive)
try:
    from google.colab import files
    files.download(archive)
except ImportError:
    print('Local run:', RUN_DIR)


## Decision record and limitations

For a decision or deliberate no-action, record the evidence, applicable policy, alternatives, costs, funding, portfolio effects, and thesis-invalidating conditions separately. AI assistance was used in development and documentation; investment judgment remains with the participant.

The historical holdout is reused research, not a prospective performance test. Labels and funds overlap; many symbols, horizons, and models are examined. No multiple-testing correction, uncertainty interval, costed strategy backtest, slippage model, probability calibration, or live execution is implemented. The heuristic gate uses holdout outcomes and needs fresh prospective validation.

**No transactions are executed by this notebook.**
